### **Adaptadores**

Este cuaderno explica qué es un adaptador, cómo se implementa en PyTorch, cómo se inserta en capas lineales y por qué reduce el número de parámetros entrenables sin modificar todo el modelo.

#### **Configuración**

#### **Objetivo**

Al terminar este cuaderno deberías poder:

1. explicar la idea de un adaptador como un cuello de botella residual,
2. insertar adaptadores en capas lineales,
3. congelar el modelo base y entrenar solo adaptadores y clasificador,
4. comparar parámetros totales frente a parámetros entrenables.

#### **Importar librerías**

Usaremos solo PyTorch y utilidades básicas.

In [ ]:
import copy
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

#### **Funciones auxiliares**

Dos funciones simples bastan para este cuaderno: contar parámetros y ejecutar una época de entrenamiento o evaluación.

In [ ]:
def count_parameters(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    trainable = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, trainable, 100 * trainable / total

def run_epoch(modelo, dataloader, criterion, optimizer=None, device=device):
    training = optimizer is not None
    modelo.train(training)
    total_loss, total_correct, total_examples = 0.0, 0, 0

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)

        logits = modelo(xb)
        loss = criterion(logits, yb)

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_examples += xb.size(0)

    return total_loss / total_examples, total_correct / total_examples

#### **¿Qué hace un adaptador?**

Si una capa produce una representación $h \in \mathbb{R}^{d}$, un adaptador inserta una transformación pequeña con cuello de botella:

$$
z = W_{\text{down}} h, \qquad
u = \phi(z), \qquad
a = W_{\text{up}} u
$$

y luego la añade de forma residual:

$$
h' = h + a
$$

donde $W_{\text{down}} \in \mathbb{R}^{r \times d}$, $W_{\text{up}} \in \mathbb{R}^{d \times r}$ y $r \ll d$.

La idea es simple: el modelo base se congela y solo se aprenden correcciones pequeñas pero útiles.

#### **Implementación del módulo adaptador**

Primero definimos un adaptador residual. Luego envolvemos una capa lineal para sumar esa corrección a su salida.

In [ ]:
class FeatureAdapter(nn.Module):
    def __init__(self, dim, bottleneck=16, activation=nn.ReLU()):
        super().__init__()
        self.down = nn.Linear(dim, bottleneck, bias=False)
        self.up = nn.Linear(bottleneck, dim, bias=False)
        self.activation = activation

    def forward(self, x):
        return x + self.up(self.activation(self.down(x)))


class AdaptedLinear(nn.Module):
    def __init__(self, linear_layer, bottleneck=16):
        super().__init__()
        self.linear = linear_layer
        self.adapter = FeatureAdapter(linear_layer.out_features, bottleneck=bottleneck)

    def forward(self, x):
        y = self.linear(x)
        return self.adapter(y)

#### **Prueba rápida sobre una capa lineal**

Aquí solo verificamos dimensiones y número de parámetros.

In [ ]:
base_linear = nn.Linear(32, 64)
adapted_linear = AdaptedLinear(copy.deepcopy(base_linear), bottleneck=8)

x = torch.randn(4, 32)
y = adapted_linear(x)

print("Salida:", y.shape)
print("Parámetros de la capa base:", sum(p.numel() for p in base_linear.parameters()))
print("Parámetros de la versión adaptada:", sum(p.numel() for p in adapted_linear.parameters()))

### **Red base y versión con adaptadores**

#### **Modelo base**

Usaremos una red pequeña para ver el mecanismo sin distraernos con un pipeline largo.

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=64, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

base_model = SmallMLP().to(device)
base_model

#### **Insertar adaptadores en capas lineales**

La siguiente función recorre el modelo y reemplaza las capas lineales seleccionadas por versiones adaptadas.

In [ ]:
def add_adapters(module, bottleneck=8, skip_names=("classifier",)):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name not in skip_names:
            setattr(module, name, AdaptedLinear(copy.deepcopy(child), bottleneck=bottleneck))
        else:
            add_adapters(child, bottleneck=bottleneck, skip_names=skip_names)
    return module

adapted_model = add_adapters(copy.deepcopy(base_model), bottleneck=8).to(device)
adapted_model

#### **Congelar el modelo base**

En PEFT se suele congelar el modelo original y entrenar solo adaptadores y capa final.

In [ ]:
for param in adapted_model.parameters():
    param.requires_grad = False

for module in adapted_model.modules():
    if isinstance(module, FeatureAdapter):
        for p in module.parameters():
            p.requires_grad = True

for p in adapted_model.classifier.parameters():
    p.requires_grad = True

count_parameters(base_model), count_parameters(adapted_model)

#### **Comparación de parámetros**

La segunda tupla muestra que la fracción entrenable baja de forma importante.

In [ ]:
base_total, base_trainable, base_pct = count_parameters(base_model)
adapt_total, adapt_trainable, adapt_pct = count_parameters(adapted_model)

print(f"Modelo base     -> total: {base_total:,} | entrenables: {base_trainable:,} | % entrenable: {base_pct:.2f}")
print(f"Modelo adaptado -> total: {adapt_total:,} | entrenables: {adapt_trainable:,} | % entrenable: {adapt_pct:.2f}")

### **Entrenamiento mínimo**

#### **Datos sintéticos**

Construimos un problema sencillo de clasificación binaria solo para comprobar que el flujo de entrenamiento funciona.

In [ ]:
n_train, n_valid, input_dim = 1024, 256, 20

X_train = torch.randn(n_train, input_dim)
X_valid = torch.randn(n_valid, input_dim)

true_w = torch.randn(input_dim, 1)
train_score = X_train @ true_w + 0.25 * torch.randn(n_train, 1)
valid_score = X_valid @ true_w + 0.25 * torch.randn(n_valid, 1)

y_train = (train_score.squeeze() > 0).long()
y_valid = (valid_score.squeeze() > 0).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=128)

#### **Entrenar solo adaptadores y clasificador**

No buscamos récords de exactitud. Solo verificamos que el esquema PEFT sea claro y ejecutable.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    [p for p in adapted_model.parameters() if p.requires_grad],
    lr=1e-3
)

history = []
for epoch in range(1, 11):
    train_loss, train_acc = run_epoch(adapted_model, train_loader, criterion, optimizer=optimizer)
    valid_loss, valid_acc = run_epoch(adapted_model, valid_loader, criterion, optimizer=None)
    history.append((epoch, train_loss, train_acc, valid_loss, valid_acc))

history[-3:]

#### **Resumen del entrenamiento**

Mira la tendencia general de pérdida y exactitud.

In [ ]:
for epoch, train_loss, train_acc, valid_loss, valid_acc in history:
    print(
        f"época {epoch:02d} | "
        f"train loss {train_loss:.4f} | train acc {train_acc:.3f} | "
        f"valid loss {valid_loss:.4f} | valid acc {valid_acc:.3f}"
    )

### **Conclusiones**

#### **Qué debes recordar**

1. Un adaptador aprende una corrección residual de baja dimensión.
2. El modelo base puede congelarse casi por completo.
3. Se reducen mucho los parámetros entrenables.
4. La idea es útil cuando el modelo completo sería caro de ajustar.

#### **Cuándo usar adaptadores**

Úsalos cuando quieras:

- reducir memoria de entrenamiento,
- conservar el modelo base,
- mantener una ruta clara de PEFT,
- experimentar con varios ajustes sin duplicar un modelo completo.

#### **Limitación importante**

Reducir parámetros entrenables no garantiza mejor desempeño. El cuello de botella $r$, las capas adaptadas y la calidad del dataset siguen siendo decisivos.

### **Ejercicios**

#### **Ejercicios básicos**

1. Cambia `bottleneck=8` por `4`, `16` y `32`. ¿Cómo cambia el porcentaje de parámetros entrenables?
2. Adapta solo la primera capa lineal de `features`. ¿Qué cambia en el conteo de parámetros?
3. Quita la conexión residual en `FeatureAdapter`. ¿Qué efecto notas en entrenamiento?
4. Sustituye `ReLU` por `GELU`. ¿Cambia algo en la estabilidad del entrenamiento?

#### **Ejercicios de análisis**

5. Explica en tus palabras la diferencia entre:
   - fine tuning completo,
   - entrenar solo la capa final,
   - usar adaptadores.

6. ¿Por qué los adaptadores encajan dentro de PEFT?

7. Si tuvieras un modelo muy grande y poca GPU, ¿por qué preferirías adaptadores frente a ajuste completo?

#### **Ejercicios de sustentación**

8. Explica la fórmula $h' = h + W_{up}\phi(W_{down}h)$.
9. ¿Qué representa el cuello de botella $r$?
10. ¿Qué parámetros quedan entrenables en este cuaderno?
11. ¿Cuál es el trade off principal entre eficiencia y capacidad de ajuste?